# Example: SharePoint ACL Ingestion — Libraries, Lists, and ASPX Pages

This notebook provisions a full chunked-vectorization pipeline on Foundry IQ (Azure AI Search) that indexes all SharePoint content types (document libraries, custom lists, and ASPX site pages) from a single `allSiteContent` data source while ingesting per-document access control lists (ACLs).

**Pre-requirements:**
1. Follow the steps to set up the registered application to allow [SharePoint indexing in Foundry IQ](https://learn.microsoft.com/azure/search/search-how-to-index-sharepoint-online).

**Flow:**
1. Create a chunks index with `UserIds`/`GroupIds` ACL fields (`permissionFilterOption = enabled`), vector search, semantic ranking, and custom SharePoint list-column fields.
2. Create a SharePoint data source targeting `allSiteContent` with `indexerPermissionOptions = [userIds, groupIds]`.
3. Create a skillset with `SplitSkill` (chunking) and `AzureOpenAIEmbeddingSkill` (vectorization). Index projections propagate ACL values and list-column metadata from the parent document to every chunk.
4. Create and run the indexer with field mappings for `metadata_spo_site_asset_item_id` (the key for mixed-content sources) and any custom list columns.
5. Query the index with a caller token in `x-ms-query-source-authorization` — the service trims results to only documents the caller can access.

**SDK:** `azure-search-documents==12.1.0b1` (REST API `2026-05-01-preview`)

**App-registration permissions required (Microsoft Graph, application):**
- `Files.Read.All`
- `Sites.FullControl.All` (or `Sites.Selected` scoped to the target site)
- Admin consent granted for both permissions

Save `sample.env` as `.env` and fill in the values, then create a virtual environment from `requirements.txt`.

## Load connections

Before running this cell, copy `sample.env` to `.env` and fill in the values.

In [ ]:
import os
import time

from azure.identity import ClientSecretCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.models import VectorizableTextQuery
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint      = os.environ['AZURE_SEARCH_ENDPOINT']
client_id     = os.environ['AZURE_CLIENT_ID']
client_secret = os.environ['AZURE_CLIENT_SECRET']
tenant_id     = os.environ['AZURE_TENANT_ID']
sp_site_url   = os.environ['SHAREPOINT_SITE_URL']

index_name    = os.getenv('AZURE_SEARCH_SP_INDEX_NAME',    'purview-sp-acl-chunks-idx')
ds_name       = os.getenv('AZURE_SEARCH_SP_DS_NAME',       'purview-sp-acl-ds')
skillset_name = os.getenv('AZURE_SEARCH_SP_SKILLSET_NAME', 'purview-sp-acl-skillset')
indexer_name  = os.getenv('AZURE_SEARCH_SP_INDEXER_NAME',  'purview-sp-acl-ixr')

openai_endpoint             = os.environ['AZURE_OPENAI_ENDPOINT']
openai_api_key              = os.environ['AZURE_OPENAI_API_KEY']
openai_embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-small')
openai_embedding_model      = os.getenv('AZURE_OPENAI_EMBEDDING_MODEL',      'text-embedding-3-small')
openai_embedding_dimensions = int(os.getenv('AZURE_OPENAI_EMBEDDING_DIMENSIONS', '1536'))

# Comma-separated list of SharePoint list columns to surface in the index.
# Customize to match the columns defined on your target SharePoint list(s).
list_columns = [
    c.strip()
    for c in os.getenv('SHAREPOINT_LIST_COLUMNS', 'Title,Category,Status,Priority').split(',')
    if c.strip()
]

credential     = ClientSecretCredential(
    tenant_id=tenant_id, client_id=client_id, client_secret=client_secret
)
index_client   = SearchIndexClient(endpoint=endpoint, credential=credential)
indexer_client = SearchIndexerClient(endpoint=endpoint, credential=credential)
search_client  = SearchClient(endpoint=endpoint, index_name=index_name, credential=credential)

sp_connection_string = (
    f'SharePointOnlineEndpoint={sp_site_url};'
    f'ApplicationId={client_id};'
    f'ApplicationSecret={client_secret};'
    f'TenantId={tenant_id}'
)

print(f'Endpoint:     {endpoint}')
print(f'Index:        {index_name}')
print(f'Data source:  {ds_name}')
print(f'Skillset:     {skillset_name}')
print(f'Indexer:      {indexer_name}')
print(f'List columns: {list_columns}')

## Create the chunks index

A single index stores every chunk produced by the skillset. Key design decisions:

- `permissionFilterOption = enabled` — activates ACL-based trimming at query time.
- `UserIds` / `GroupIds` — `Collection(Edm.String)` fields annotated with `permissionFilter`. The service filters out any chunk whose ACL does not include the caller's identity. Set `retrievable=False` in production.
- `parent_id` — carries the base64-encoded `metadata_spo_site_asset_item_id` of the source document/list item/page. Used to join chunks back to their parent.
- `metadata_spo_item_content_type` — lets you distinguish library documents (`application/pdf` etc.), list items (`text/html`), and ASPX pages in query results.
- List-column fields (`list_*`) — one `Edm.String` field per column in `SHAREPOINT_LIST_COLUMNS`. Every chunk inherits the full list-row metadata via index projections.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    HnswAlgorithmConfiguration,
    SearchField,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    VectorSearch,
    VectorSearchProfile,
)

# Dynamically build one Edm.String field per configured list column.
list_column_fields = [
    SearchField(
        name=f'list_{col.lower()}',
        type='Edm.String',
        searchable=True,
        filterable=True,
        facetable=True,
        retrievable=True,
    )
    for col in list_columns
]

index = SearchIndex(
    name=index_name,
    fields=[
        # Chunk identity
        SearchField(name='id',        type='Edm.String',          key=True, filterable=True),
        SearchField(name='parent_id', type='Edm.String',          filterable=True, sortable=True,  retrievable=True),
        # Chunk text and embedding
        SearchField(name='content',   type='Edm.String',          searchable=True, retrievable=True),
        SearchField(
            name='content_vector',
            type='Collection(Edm.Single)',
            searchable=True,
            stored=False,
            vector_search_dimensions=openai_embedding_dimensions,
            vector_search_profile_name='hnsw-profile',
        ),
        # SharePoint metadata surfaced for all content types
        SearchField(name='metadata_spo_item_name',           type='Edm.String',          searchable=True, filterable=True, sortable=True,  retrievable=True),
        SearchField(name='metadata_spo_item_weburi',         type='Edm.String',          filterable=True,                                  retrievable=True),
        SearchField(name='metadata_spo_item_last_modified',  type='Edm.DateTimeOffset',  filterable=True, sortable=True,                   retrievable=True),
        # content_type distinguishes: library docs, list items (text/html), ASPX pages
        SearchField(name='metadata_spo_item_content_type',   type='Edm.String',          filterable=True, facetable=True,                  retrievable=True),
        # ACL permission fields — set retrievable=False before publishing to production
        SearchField(name='UserIds',  type='Collection(Edm.String)', filterable=True, retrievable=True, permission_filter='userIds'),
        SearchField(name='GroupIds', type='Collection(Edm.String)', filterable=True, retrievable=True, permission_filter='groupIds'),
        # List-column fields (one per SHAREPOINT_LIST_COLUMNS entry)
        *list_column_fields,
    ],
    # Enable ACL-based result trimming at query time
    permission_filter_option='enabled',
    vector_search=VectorSearch(
        profiles=[VectorSearchProfile(
            name='hnsw-profile',
            algorithm_configuration_name='hnsw',
            vectorizer_name='openai-vec',
        )],
        algorithms=[HnswAlgorithmConfiguration(name='hnsw')],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name='openai-vec',
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=openai_endpoint,
                    deployment_name=openai_embedding_deployment,
                    model_name=openai_embedding_model,
                    api_key=openai_api_key,
                ),
            )
        ],
    ),
    semantic_search=SemanticSearch(
        default_configuration_name='semantic-config',
        configurations=[
            SemanticConfiguration(
                name='semantic-config',
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name='metadata_spo_item_name'),
                    content_fields=[SemanticField(field_name='content')],
                ),
            )
        ],
    ),
)

idx = index_client.create_or_update_index(index)
acl_fields = [f.name for f in idx.fields if getattr(f, 'permission_filter', None)]
print(
    f"Index '{idx.name}' ready  "
    f"({len(idx.fields)} fields | "
    f"ACL fields: {acl_fields} | "
    f"permissionFilterOption={idx.permission_filter_option})"
)

## Create the SharePoint data source

Using the `allSiteContent` container indexes three content types in a single indexer run:

| Content type | Key source field | Example `metadata_spo_item_content_type` |
|---|---|---|
| Document library files | `metadata_spo_site_asset_item_id` | `application/pdf`, `application/vnd.openxmlformats-officedocument…` |
| SharePoint list items | `metadata_spo_site_asset_item_id` | `text/html` |
| ASPX site pages | `metadata_spo_site_asset_item_id` | `text/html` |

`indexerPermissionOptions: [userIds, groupIds]` instructs the indexer to read each item's SharePoint permissions and store them in the enriched document at `/document/metadata_user_ids` and `/document/metadata_group_ids`.

`additionalColumns` makes custom SharePoint list columns available as source fields by their column name.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection,
)

container_query_parts = ['includeSubsites=true']
if list_columns:
    container_query_parts.append(f"additionalColumns={','.join(list_columns)}")
container_query = ';'.join(container_query_parts)

ds = SearchIndexerDataSourceConnection(
    name=ds_name,
    type='sharepoint',
    connection_string=sp_connection_string,
    container=SearchIndexerDataContainer(name='allSiteContent', query=container_query),
    # Instruct the indexer to read and store SharePoint user and group ACLs
    indexer_permission_options=['userIds', 'groupIds'],
)

created_ds = indexer_client.create_or_update_data_source_connection(ds)
print(
    f"Data source '{created_ds.name}' ready  "
    f"(container='{created_ds.container.name}'  "
    f"query='{created_ds.container.query}'  "
    f"permissionOptions={created_ds.indexer_permission_options})"
)

## Create the skillset with chunking, embedding, and index projections

**Skills:**
- `SplitSkill` — splits each document's `content` into 2 000-character pages with 200-character overlap, producing an array at `/document/chunks/*`.
- `AzureOpenAIEmbeddingSkill` — embeds each chunk individually; runs in the context of `/document/chunks/*` so one embedding is produced per chunk.

**Index projections:**
Each projection selector writes one row per chunk to the target index. The `mappings` list determines which enriched-document paths end up in which index fields:

| Index field | Enriched document path | Notes |
|---|---|---|
| `content` | `/document/chunks/*` | chunk text |
| `content_vector` | `/document/chunks/*/embedding` | chunk embedding |
| `UserIds` | `/document/metadata_user_ids` | **ACL propagation** — every chunk inherits the parent's user list |
| `GroupIds` | `/document/metadata_group_ids` | **ACL propagation** — every chunk inherits the parent's group list |
| `list_<col>` | `/document/<Col>` | List column value copied to every chunk from the parent row |

`projectionMode: skipIndexingParentDocuments` prevents the un-chunked parent document from being written to the index.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIEmbeddingSkill,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    SearchIndexerIndexProjections,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    SearchIndexerSkillset,
    SplitSkill,
)

split_skill = SplitSkill(
    name='split',
    description='Split content into overlapping pages for chunk-level vectorization',
    context='/document',
    text_split_mode='pages',
    maximum_page_length=2000,
    page_overlap_length=200,
    inputs=[InputFieldMappingEntry(name='text', source='/document/content')],
    outputs=[OutputFieldMappingEntry(name='textItems', target_name='chunks')],
)

embedding_skill = AzureOpenAIEmbeddingSkill(
    name='embedding',
    description='Embed each text chunk with Azure OpenAI',
    context='/document/chunks/*',
    resource_uri=openai_endpoint,
    api_key=openai_api_key,
    deployment_id=openai_embedding_deployment,
    model_name=openai_embedding_model,
    dimensions=openai_embedding_dimensions,
    inputs=[InputFieldMappingEntry(name='text', source='/document/chunks/*')],
    outputs=[OutputFieldMappingEntry(name='embedding', target_name='embedding')],
)

# Fixed projection mappings: chunk fields + SharePoint metadata + ACL fields
fixed_projection_mappings = [
    InputFieldMappingEntry(name='content',                       source='/document/chunks/*'),
    InputFieldMappingEntry(name='content_vector',                source='/document/chunks/*/embedding'),
    InputFieldMappingEntry(name='metadata_spo_item_name',        source='/document/metadata_spo_item_name'),
    InputFieldMappingEntry(name='metadata_spo_item_weburi',      source='/document/metadata_spo_item_weburi'),
    InputFieldMappingEntry(name='metadata_spo_item_last_modified', source='/document/metadata_spo_item_last_modified'),
    InputFieldMappingEntry(name='metadata_spo_item_content_type',  source='/document/metadata_spo_item_content_type'),
    # ACL propagation: these paths are populated by the indexer when indexerPermissionOptions
    # includes 'userIds' and 'groupIds'. Every chunk inherits the parent document's ACL.
    InputFieldMappingEntry(name='UserIds',  source='/document/metadata_user_ids'),
    InputFieldMappingEntry(name='GroupIds', source='/document/metadata_group_ids'),
]

# Dynamic list-column projection mappings: each chunk carries the full list-row metadata.
list_col_projection_mappings = [
    InputFieldMappingEntry(name=f'list_{col.lower()}', source=f'/document/{col}')
    for col in list_columns
]

index_projections = SearchIndexerIndexProjections(
    selectors=[
        SearchIndexerIndexProjectionSelector(
            target_index_name=index_name,
            parent_key_field_name='parent_id',
            source_context='/document/chunks/*',
            mappings=fixed_projection_mappings + list_col_projection_mappings,
        )
    ],
    parameters=SearchIndexerIndexProjectionsParameters(
        projection_mode='skipIndexingParentDocuments',
    ),
)

skillset = SearchIndexerSkillset(
    name=skillset_name,
    description='Chunk, embed, and propagate ACLs for allSiteContent (libraries + lists + ASPX pages)',
    skills=[split_skill, embedding_skill],
    index_projections=index_projections,
)

ss = indexer_client.create_or_update_skillset(skillset)
print(
    f"Skillset '{ss.name}' ready  "
    f"({len(ss.skills)} skills | "
    f"{len(ss.index_projections.selectors)} projection selector | "
    f"{len(ss.index_projections.selectors[0].mappings)} mapped fields)"
)

## Create and run the indexer

The only required field mapping is:

- `metadata_spo_site_asset_item_id → parent_id` (base64Encode) — `allSiteContent` uses this field (not `metadata_spo_site_library_item_id`) as the unique key across all content types.

SharePoint list columns (`Title`, `Price`, `InStock`, `Category`, etc.) are surfaced by the indexer as direct source fields on the enriched document — no explicit field mapping is needed. The skillset’s index projections propagate them from `/document/<Col>` to the `list_<col>` index field on every chunk.

`fail_on_unsupported_content_type=False` and `fail_on_unprocessable_document=False` prevent binary files such as `.zip` or `.exe` from halting the run.

In [ ]:
from azure.search.documents.indexes.models import (
    FieldMapping,
    FieldMappingFunction,
    IndexingParameters,
    IndexingParametersConfiguration,
    SearchIndexer,
)

indexer = SearchIndexer(
    name=indexer_name,
    data_source_name=ds_name,
    target_index_name=index_name,
    skillset_name=skillset_name,
    parameters=IndexingParameters(
        configuration=IndexingParametersConfiguration(
            data_to_extract='contentAndMetadata',
            fail_on_unsupported_content_type=False,
            fail_on_unprocessable_document=False,
        )
    ),
    field_mappings=[
        # allSiteContent requires base64Encode on the asset item ID to produce a valid key.
        # SharePoint list columns surface as direct source fields on the enriched document
        # (no mapping needed here); the skillset's index projections propagate them
        # from /document/<Col> to list_<col> on every chunk.
        FieldMapping(
            source_field_name='metadata_spo_site_asset_item_id',
            target_field_name='parent_id',
            mapping_function=FieldMappingFunction(name='base64Encode'),
        ),
    ],
)

indexer_client.create_or_update_indexer(indexer)
indexer_client.run_indexer(indexer_name)
print(f"Indexer '{indexer_name}' created and started")

## Poll indexer status

Polls silently every 30 seconds and prints only the terminal result. Initial runs over large SharePoint sites may take several minutes.

In [ ]:
POLL_INTERVAL_SECS = 30
POLL_TIMEOUT_SECS  = 1800  # 30 minutes

start = time.time()
while True:
    status_obj = indexer_client.get_indexer_status(indexer_name)
    lr = status_obj.last_result
    if lr and getattr(lr, 'status', None) in ('success', 'error', 'transientFailure'):
        break
    if time.time() - start > POLL_TIMEOUT_SECS:
        break
    time.sleep(POLL_INTERVAL_SECS)

lr = indexer_client.get_indexer_status(indexer_name).last_result
print(
    f"Indexer '{indexer_name}'  "
    f"status={getattr(lr, 'status', 'unknown')}  "
    f"items_processed={getattr(lr, 'item_count', 0)}  "
    f"failed={getattr(lr, 'failed_item_count', 0)}"
)

## Query with ACL filtering

Pass a raw Azure AI Search access token in `x-ms-query-source-authorization` (no `Bearer` prefix). The service intersects the caller's identity against each chunk's `UserIds`/`GroupIds` fields and trims the result set to only the chunks the caller has permission to read.

The query uses semantic hybrid search (keyword + vector), so the vectorizer on the index automatically embeds the query text — no separate embedding call needed.

Results show:
- `metadata_spo_item_name` — name of the source document, list item, or ASPX page
- `metadata_spo_item_content_type` — identifies whether the source was a library file, list item, or page
- `UserIds` / `GroupIds` — ACL values stored on each chunk (visible here for verification)
- `list_*` fields — SharePoint list-column values propagated to every chunk from the parent list item

In [ ]:
# Acquire a raw Search access token — do NOT include a 'Bearer' prefix
raw_token = credential.get_token('https://search.azure.com/.default').token

query_text = 'documents files site pages'

results = list(
    search_client.search(
        search_text=query_text,
        query_type='semantic',
        semantic_configuration_name='semantic-config',
        vector_queries=[
            VectorizableTextQuery(
                text=query_text,
                k_nearest_neighbors=5,
                fields='content_vector',
            )
        ],
        select=[
            'metadata_spo_item_name',
            'metadata_spo_item_weburi',
            'metadata_spo_item_content_type',
            'content',
            'UserIds',
            'GroupIds',
            *[f'list_{col.lower()}' for col in list_columns],
        ],
        top=10,
        # Permission-filtered retrieval: only chunks the caller can access are returned
        headers={'x-ms-query-source-authorization': raw_token},
    )
)

print(f"{len(results)} chunk(s) returned (ACL-filtered for caller)\n")
w = 42
print(f"{'Item name':<{w}}  {'Content type':<38}  {'GroupIds':<28}  Content excerpt")
print('-' * 145)
for r in results:
    name    = (r.get('metadata_spo_item_name')         or '')[:w]
    ctype   = (r.get('metadata_spo_item_content_type') or '')[:38]
    groups  = str(r.get('GroupIds') or [])[:28]
    excerpt = (r.get('content') or '').replace('\n', ' ')[:70]
    print(f"  {name:<{w}}  {ctype:<38}  {groups:<28}  {excerpt}...")
    list_vals = {
        f'list_{col.lower()}': r.get(f'list_{col.lower()}')
        for col in list_columns
        if r.get(f'list_{col.lower()}')
    }
    if list_vals:
        print(f"    List columns: {list_vals}")

## Clean up

Delete resources in dependency order: indexer → skillset → index → data source.

In [ ]:
indexer_client.delete_indexer(indexer_name)
print(f"Indexer '{indexer_name}' deleted")

In [ ]:
indexer_client.delete_skillset(skillset_name)
print(f"Skillset '{skillset_name}' deleted")

In [ ]:
index_client.delete_index(index_name)
print(f"Index '{index_name}' deleted")

In [ ]:
indexer_client.delete_data_source_connection(ds_name)
print(f"Data source '{ds_name}' deleted")